# Clasificacion de Frutas con Q-Learning + Simulacion visual

Una banda clasifica frutas en tres tipos y las envia a tres destinos:
**SANO -> VENTA**, **APROVECHABLE -> PROCESO** (jugo/mermelada), **PODRIDO -> DESECHABLE**.

Cada parte esta separada en su propia seccion: librerias, configuracion, entorno,
entrenamiento y luego la simulacion en pygame (dividida tambien por secciones).

**Como correrlo:** ejecuta las secciones 1 a 5 una vez (entrenan y guardan
`q_tabla.npy`). Luego ejecuta las secciones 6 a 11 para ver la simulacion.
La ventana de pygame necesita una PC con escritorio (no funciona en servidores
sin pantalla).

## Seccion 1 - Librerias

In [12]:
import os
import numpy as np
import pygame

## Seccion 2 - Configuracion general

Probabilidades de cada producto, nombres legibles y la **matriz de recompensas**
(filas = estado, columnas = accion). La diagonal es la decision correcta.

In [13]:
# Probabilidades de aparicion: bueno, reparable, malo (deben sumar 1.0)
PROB = [0.5, 0.2, 0.3]
PROB_BUENO, PROB_REPARABLE, PROB_MALO = PROB
assert abs(sum(PROB) - 1.0) < 1e-9

NOMBRE_ESTADO = {0: "SANO", 1: "APROVECHABLE", 2: "PODRIDO"}
SIGNIFICADO   = {0: "Producto sano, listo para la venta",
                 1: "Golpeado o muy maduro: se aprovecha (jugo, mermelada)",
                 2: "Producto podrido, no se puede usar"}
NOMBRE_ACCION = {0: "dejar pasar", 1: "aprovechar", 2: "desechar"}

# Matriz de recompensas: filas = estado, columnas = accion
RECOMPENSAS = np.array([
    #  dejar  reparar  desechar
    [    5,    -3,      -5 ],   # bueno
    [   -8,     8,      -2 ],   # reparable
    [  -15,    -5,      10 ],   # malo
], dtype=np.float64)

N_ESTADOS, N_ACCIONES = RECOMPENSAS.shape

## Seccion 3 - Entorno (cinta transportadora)

En cada ronda aparece un producto al azar segun las probabilidades. La recompensa
se lee directo de la matriz `RECOMPENSAS`.

In [14]:
class CintaTransportadoraEnv:
    def __init__(self, prob_malo=PROB_MALO, prob_reparable=PROB_REPARABLE, semilla=None):
        self.prob_malo = prob_malo
        self.prob_reparable = prob_reparable
        self.prob_bueno = 1.0 - prob_malo - prob_reparable
        assert self.prob_bueno >= 0, "Las probabilidades suman mas de 1.0"
        self.rng = np.random.default_rng(semilla)
        self.producto_actual = None

    def reiniciar(self):
        self.producto_actual = int(self.rng.choice(
            [0, 1, 2], p=[self.prob_bueno, self.prob_reparable, self.prob_malo]))
        return self.producto_actual

    def paso(self, accion):
        prod = self.producto_actual
        recompensa = float(RECOMPENSAS[prod, accion])
        siguiente = self.reiniciar()
        return siguiente, recompensa, False, {"producto": prod}

## Seccion 4 - Algoritmo Q-Learning

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma\,\max_{a'}Q(s',a') - Q(s,a)\,]$$

In [15]:
def entrenamiento(episodios, alpha, gamma, epsilon, decay_epsilon, min_epsilon,
                  prob_reparable, prob_malo, semilla, guardar_ruta):
    env = CintaTransportadoraEnv(prob_reparable=prob_reparable,
                                 prob_malo=prob_malo, semilla=semilla)
    rng = np.random.default_rng(semilla)
    Q = np.zeros((N_ESTADOS, N_ACCIONES), dtype=np.float64)
    historial = []
    eps = epsilon

    for ep in range(episodios):
        estado = env.reiniciar()
        if rng.random() < eps:
            accion = int(rng.integers(N_ACCIONES))   # explorar
        else:
            accion = int(np.argmax(Q[estado]))        # explotar

        siguiente, recompensa, _, _ = env.paso(accion)
        td_objetivo = recompensa + gamma * np.max(Q[siguiente])
        Q[estado, accion] += alpha * (td_objetivo - Q[estado, accion])

        historial.append(recompensa)
        eps = max(min_epsilon, eps * decay_epsilon)
        if (ep + 1) % 2000 == 0:
            prom = np.mean(historial[-2000:])
            print(f"Episodio {ep+1}/{episodios} | recompensa_prom: {prom:.2f} | eps: {eps:.4f}")

    np.save(guardar_ruta, Q)
    print(f"Entrenamiento finalizado. Q-tabla guardada en: {guardar_ruta}")
    return Q, historial

## Seccion 5 - Entrenar el agente

In [16]:
Q, recompensas = entrenamiento(
    episodios=20000, alpha=0.2, gamma=0.99,
    epsilon=0.1, decay_epsilon=0.9998, min_epsilon=0.01,
    prob_reparable=PROB_REPARABLE, prob_malo=PROB_MALO,
    semilla=123, guardar_ruta="q_tabla.npy")

print("\nPolitica aprendida:")
for s in range(N_ESTADOS):
    print(f"  {NOMBRE_ESTADO[s]:>9} -> {NOMBRE_ACCION[int(np.argmax(Q[s]))]}")

Episodio 2000/20000 | recompensa_prom: 6.28 | eps: 0.0670
Episodio 4000/20000 | recompensa_prom: 6.55 | eps: 0.0449
Episodio 6000/20000 | recompensa_prom: 6.66 | eps: 0.0301
Episodio 8000/20000 | recompensa_prom: 6.90 | eps: 0.0202
Episodio 10000/20000 | recompensa_prom: 7.02 | eps: 0.0135
Episodio 12000/20000 | recompensa_prom: 6.91 | eps: 0.0100
Episodio 14000/20000 | recompensa_prom: 7.13 | eps: 0.0100
Episodio 16000/20000 | recompensa_prom: 7.01 | eps: 0.0100
Episodio 18000/20000 | recompensa_prom: 6.96 | eps: 0.0100
Episodio 20000/20000 | recompensa_prom: 7.05 | eps: 0.0100
Entrenamiento finalizado. Q-tabla guardada en: q_tabla.npy

Politica aprendida:
       SANO -> dejar pasar
  APROVECHABLE -> aprovechar
    PODRIDO -> desechar


## Seccion 6 - Configuracion visual de la simulacion

Colores, medidas de la ventana y posicion de los tres contenedores.

In [17]:
ANCHO, ALTO = 980, 600
FONDO=(245,247,250); PANEL=(255,255,255); BORDE=(220,224,230)
TEXTO=(35,40,50); SUAVE=(120,128,140); CINTA=(105,112,124)
RESALTE=(60,130,220); ROJO_ERR=(235,70,70)
VERDE=(46,184,92); AMBAR=(240,190,50); ROJO=(210,70,70)

COLOR_ESTADO = {0:VERDE, 1:AMBAR, 2:ROJO}
CONTENEDOR   = {0:("VENTA",VERDE), 1:("PROCESO",AMBAR), 2:("DESECHABLE",ROJO)}

X_PANEL=300; Y_CINTA=330; X_BIFURCA=760; X_BINS=845; X_SENSOR=600
Y_BIN={1:170, 0:Y_CINTA, 2:490}   # reparar arriba, salida centro, desecho abajo

## Seccion 7 - Dibujo: cinta, contenedores, sensor y brazos desviadores

El **sensor laser** (linea roja) va antes de los brazos: cuando la fruta lo cruza, da
un destello y se revela su estado (antes viaja como "?" porque aun no se conoce).
Los brazos son dos pistones: el de **arriba** empuja hacia abajo (DESECHABLE) y el de
**abajo** empuja hacia arriba (PROCESO); si la fruta esta sana, ninguno se mueve.

In [18]:
def dibujar_cinta(p, F):
    pygame.draw.rect(p, CINTA, (X_PANEL, Y_CINTA-18, X_BIFURCA-X_PANEL+40, 36), border_radius=6)
    for a in (0,1,2):
        pygame.draw.line(p, BORDE, (X_BIFURCA, Y_CINTA), (X_BINS, Y_BIN[a]), 3)

def dibujar_contenedores(p, F, conteos):
    for a,(etq,col) in CONTENEDOR.items():
        y=Y_BIN[a]; r=pygame.Rect(X_BINS-10, y-35, 105, 70)
        pygame.draw.rect(p, PANEL, r, border_radius=8)
        pygame.draw.rect(p, col, r, width=3, border_radius=8)
        p.blit(F["mini"].render(etq, True, TEXTO), (X_BINS-5, y-30))
        p.blit(F["grande"].render(str(conteos[a]), True, col), (X_BINS+20, y-8))

def dibujar_leyenda(p, F):
    x,y=X_PANEL+10, ALTO-40
    for e in (0,1,2):
        pygame.draw.circle(p, COLOR_ESTADO[e], (x,y), 9)
        p.blit(F["chico"].render(NOMBRE_ESTADO[e], True, TEXTO), (x+16, y-8)); x+=150

def dibujar_brazos(p, F, ext_sup, ext_inf):
    arm_x = X_BIFURCA - 16; ancho = 30; METAL = (150,158,170)
    # brazo superior: empuja hacia ABAJO -> DESECHO (se ilumina en rojo)
    pygame.draw.rect(p, METAL, (arm_x-3, Y_CINTA-100, ancho+6, 12), border_radius=3)
    head_sup = (Y_CINTA-78) + ext_sup*48
    col_s = ROJO if ext_sup > 0.05 else METAL
    pygame.draw.rect(p, col_s, (arm_x+ancho//2-6, Y_CINTA-88, 12, head_sup-(Y_CINTA-88)), border_radius=3)
    pygame.draw.rect(p, col_s, (arm_x, head_sup, ancho, 16), border_radius=4)
    # brazo inferior: empuja hacia ARRIBA -> REPARACION (se ilumina en amarillo)
    pygame.draw.rect(p, METAL, (arm_x-3, Y_CINTA+88, ancho+6, 12), border_radius=3)
    head_inf = (Y_CINTA+62) - ext_inf*48
    col_i = AMBAR if ext_inf > 0.05 else METAL
    pygame.draw.rect(p, col_i, (arm_x+ancho//2-6, head_inf+16, 12, (Y_CINTA+88)-(head_inf+16)), border_radius=3)
    pygame.draw.rect(p, col_i, (arm_x, head_inf, ancho, 16), border_radius=4)

def dibujar_sensor(p, F, flash):
    LASER=(255,60,60); top_y=Y_CINTA-70; bot_y=Y_CINTA+70
    pygame.draw.rect(p, (90,95,105), (X_SENSOR-9, top_y-14, 18, 14), border_radius=3)  # emisor
    pygame.draw.rect(p, (90,95,105), (X_SENSOR-9, bot_y, 18, 14), border_radius=3)     # receptor
    pygame.draw.line(p, LASER, (X_SENSOR, top_y), (X_SENSOR, bot_y), 3 + (4 if flash>0 else 0))
    p.blit(F["chico"].render("SENSOR", True, SUAVE), (X_SENSOR-24, top_y-34))
    if flash > 0:                          # destello al detectar (anillo que se expande)
        rad = int(8 + (1-flash)*30)
        aura = pygame.Surface((rad*2+6, rad*2+6), pygame.SRCALPHA)
        pygame.draw.circle(aura, (255,235,120,int(200*flash)), (rad+3,rad+3), rad, 4)
        p.blit(aura, (X_SENSOR-rad-3, Y_CINTA-rad-3))

## Seccion 8 - Dibujo: producto y panel del cerebro

El producto lleva su tipo etiquetado encima. El panel muestra los valores Q del
estado actual y resalta la accion elegida.

In [19]:
def dibujar_producto(p, F, x, y, estado, err, detectado):
    if detectado:
        col = ROJO_ERR if err else COLOR_ESTADO[estado]
        etiqueta = NOMBRE_ESTADO[estado]
    else:                                   # aun no cruza el sensor: desconocido
        col = (175,182,192); etiqueta = "?"
    pygame.draw.circle(p, col, (int(x),int(y)), 18)
    pygame.draw.circle(p, TEXTO, (int(x),int(y)), 18, 2)
    e = F["chico"].render(etiqueta, True, TEXTO)
    p.blit(e, (int(x)-e.get_width()//2, int(y)-42))

def dibujar_cerebro(p, F, Q, estado, accion, detectado):
    pygame.draw.rect(p, PANEL, (0,70,X_PANEL,ALTO-70))
    pygame.draw.line(p, BORDE, (X_PANEL,70), (X_PANEL,ALTO), 2)
    p.blit(F["medio"].render("Cerebro del agente", True, TEXTO), (20,85))
    base_y, alto_max = 360, 120
    if not detectado:                       # todavia no hay lectura del sensor
        pygame.draw.circle(p, (200,205,212), (35,140), 14)
        p.blit(F["medio"].render("Detectando...", True, SUAVE), (60,128))
        p.blit(F["chico"].render("Esperando lectura del sensor", True, SUAVE), (20,162))
        for a in (0,1,2):
            bx=40+a*80
            pygame.draw.rect(p, BORDE, (bx, base_y-30, 50, 30), border_radius=4)
            p.blit(F["chico"].render(NOMBRE_ACCION[a], True, SUAVE), (bx-4, base_y+8))
        return
    pygame.draw.circle(p, COLOR_ESTADO[estado], (35,140), 14)
    p.blit(F["medio"].render(NOMBRE_ESTADO[estado], True, TEXTO), (60,128))
    p.blit(F["chico"].render(SIGNIFICADO[estado], True, SUAVE), (20,162))
    q=Q[estado]; qmin,qmax=q.min(),q.max(); rango=(qmax-qmin) or 1.0
    for a in (0,1,2):
        bx=40+a*80; h=10+(q[a]-qmin)/rango*alto_max
        col = RESALTE if a==accion else BORDE
        pygame.draw.rect(p, col, (bx, base_y-h, 50, h), border_radius=4)
        p.blit(F["chico"].render(NOMBRE_ACCION[a], True, TEXTO), (bx-4, base_y+8))
        p.blit(F["chico"].render(f"{q[a]:.1f}", True, SUAVE), (bx+6, base_y-h-18))
    p.blit(F["medio"].render(f"Decision: {NOMBRE_ACCION[accion].upper()}", True, RESALTE),
           (20, base_y+45))

## Seccion 9 - Dibujo: HUD con estadisticas en vivo

In [20]:
def dibujar_hud(p, F, st):
    pygame.draw.rect(p, PANEL, (0,0,ANCHO,60)); pygame.draw.line(p, BORDE, (0,60),(ANCHO,60),2)
    proc=st["procesados"]; pct=(st["aciertos"]/proc*100) if proc else 0.0
    items=[f"Procesados: {proc}", f"Aciertos: {st['aciertos']}", f"Fallos: {st['fallos']}",
           f"% acierto: {pct:.1f}%", f"Recompensa: {st['recompensa']:.0f}",
           f"Agente: {'ALEATORIO' if st['aleatorio'] else 'ENTRENADO'}"]
    x=20
    for it in items:
        s=F["hud"].render(it, True, TEXTO); p.blit(s,(x,20)); x+=s.get_width()+18

## Seccion 10 - Bucle principal con controles

Controles: **ESPACIO** pausar/reanudar  ·  **N** avanzar un producto  ·
**flechas arriba/abajo** velocidad  ·  **A** alternar agente entrenado/aleatorio  ·
**R** reiniciar contadores.

In [21]:
def simular(q_ruta="q_tabla.npy"):
    if not os.path.exists(q_ruta):
        print("Q-tabla no encontrada. Entrena primero (secciones 1-5)."); return
    Q = np.load(q_ruta)
    try:
        pygame.init(); pantalla = pygame.display.set_mode((ANCHO, ALTO))
    except Exception as e:
        print("Sin pantalla disponible (entorno headless):", e); return
    pygame.display.set_caption("Cinta Transportadora - Q-Learning")
    reloj = pygame.time.Clock()
    F = {"chico":pygame.font.SysFont("Arial",15), "medio":pygame.font.SysFont("Arial",19,bold=True),
         "grande":pygame.font.SysFont("Arial",24,bold=True), "mini":pygame.font.SysFont("Arial",13),
         "hud":pygame.font.SysFont("Arial",17,bold=True)}
    rng = np.random.default_rng(7)
    st = {"procesados":0,"aciertos":0,"fallos":0,"recompensa":0.0,"aleatorio":False}
    conteos = {0:0,1:0,2:0}

    def nuevo():
        e = int(rng.choice([0,1,2], p=PROB))
        a = int(rng.integers(N_ACCIONES)) if st["aleatorio"] else int(np.argmax(Q[e]))
        return e, a
    estado, accion = nuevo()
    x, y = X_PANEL+20, Y_CINTA; vel = 3; pausado = False
    ext_sup = ext_inf = 0.0      # cuanto sobresale cada brazo (0=retraido, 1=fuera)
    detectado = False; flash = 0.0   # estado del sensor para la fruta actual

    def resolver():
        st["procesados"]+=1; conteos[accion]+=1
        st["recompensa"]+=RECOMPENSAS[estado, accion]
        if accion == int(np.argmax(RECOMPENSAS[estado])): st["aciertos"]+=1
        else: st["fallos"]+=1

    corriendo = True
    while corriendo:
        avanzar_uno = False
        for ev in pygame.event.get():
            if ev.type == pygame.QUIT: corriendo = False
            elif ev.type == pygame.KEYDOWN:
                if   ev.key == pygame.K_SPACE: pausado = not pausado
                elif ev.key == pygame.K_UP:    vel = min(12, vel+1)
                elif ev.key == pygame.K_DOWN:  vel = max(1, vel-1)
                elif ev.key == pygame.K_a:     st["aleatorio"] = not st["aleatorio"]; estado, accion = nuevo(); x, y = X_PANEL+20, Y_CINTA; detectado = False; flash = 0.0
                elif ev.key == pygame.K_r:
                    st.update(procesados=0, aciertos=0, fallos=0, recompensa=0.0); conteos.update({0:0,1:0,2:0})
                elif ev.key == pygame.K_n:     avanzar_uno = True

        # el sensor laser detecta el estado cuando la fruta lo cruza
        if not detectado and x >= X_SENSOR:
            detectado = True; flash = 1.0
        flash = max(0.0, flash - 0.06)

        # los brazos se asoman (solo si ya se detecto) cuando la fruta esta por llegar
        en_zona = (X_BIFURCA-90) <= x <= (X_BIFURCA+12)
        ext_sup += ((1.0 if accion==2 and en_zona and detectado else 0.0) - ext_sup) * 0.4
        ext_inf += ((1.0 if accion==1 and en_zona and detectado else 0.0) - ext_inf) * 0.4

        pantalla.fill(FONDO)
        dibujar_cinta(pantalla, F); dibujar_contenedores(pantalla, F, conteos)
        dibujar_sensor(pantalla, F, flash)
        dibujar_brazos(pantalla, F, ext_sup, ext_inf)
        dibujar_cerebro(pantalla, F, Q, estado, accion, detectado)
        err = (accion != int(np.argmax(RECOMPENSAS[estado])))
        parpadeo = err and detectado and (pygame.time.get_ticks()//250)%2==0
        dibujar_producto(pantalla, F, x, y, estado, parpadeo, detectado)
        dibujar_hud(pantalla, F, st); dibujar_leyenda(pantalla, F)
        pantalla.blit(F["chico"].render(
            "ESPACIO pausar  |  N siguiente  |  flechas velocidad  |  A agente  |  R reiniciar",
            True, SUAVE), (X_PANEL+20, 76))
        pygame.display.flip()

        if (not pausado) or avanzar_uno:
            x += vel; tgt = Y_BIN[accion]
            if x >= X_BIFURCA-6: y += np.sign(tgt-y) * min(vel, abs(tgt-y))
            if x >= X_BINS-10:
                resolver(); estado, accion = nuevo(); x, y = X_PANEL+20, Y_CINTA
                detectado = False; flash = 0.0
        reloj.tick(60)
    pygame.quit()

## Seccion 11 - Ejecutar la simulacion

In [22]:
simular("q_tabla.npy")